In [1]:
pip install -U torch transformers accelerate sentencepiece tqdm numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 157.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 227.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 185.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 207.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 129.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 125.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 146.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 157.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 146.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 146.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.

In [2]:
pip install python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

In [4]:
from huggingface_hub import login
login()

In [2]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip uninstall -y torchvision && pip install -U torch transformers accelerate sentencepiece tqdm numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

DEV_RATIO = 0.2
SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"
RUN_TEST = False

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    result = load_json(out_path)
    return result


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    if USE_CHAT_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def compute_steering_vectors(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    plausible_sample = balanced_by_validity(plausible, n, SEED)
    implausible_sample = balanced_by_validity(implausible, n, SEED)
    print(f"Steering: {n} plausible + {n} implausible")

    plausible_acts = {l: [] for l in target_layers}
    implausible_acts = {l: [] for l in target_layers}

    print("Extracting plausible activations...")
    for i, ex in enumerate(plausible_sample):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        for l in target_layers:
            plausible_acts[l].append(hiddens[l])
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{n}")

    print("Extracting implausible activations...")
    for i, ex in enumerate(implausible_sample):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        for l in target_layers:
            implausible_acts[l].append(hiddens[l])
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{n}")

    deltas = {}
    for l in target_layers:
        mu_p = torch.stack(plausible_acts[l]).mean(dim=0)
        mu_ip = torch.stack(implausible_acts[l]).mean(dim=0)
        delta = mu_ip - mu_p
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    return deltas


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


def register_hooks(layers, deltas, alpha):
    handles = []
    for layer_idx, delta in deltas.items():
        handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, alpha)))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset(model, tokenizer, eval_items, deltas, alpha):
    layers = get_layers(model)
    handles = []
    if deltas is not None and alpha != 0:
        handles = register_hooks(layers, deltas, alpha)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(dev_items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in dev_items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Alphas: {ALPHAS}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)
    save_json(train_items, os.path.join(OUT_DIR, "train_split.json"))
    save_json(dev_items, os.path.join(OUT_DIR, "dev_split.json"))
    print(f"Train: {len(train_items)}, Dev: {len(dev_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas = compute_steering_vectors(model, tokenizer, train_items, target_layers)
    torch.save(deltas, os.path.join(OUT_DIR, "steering_vectors.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, dev_items, deltas, alpha)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(dev_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "dev_split.json"), pred_path, os.path.join(OUT_DIR, f"eval_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print("\n===== SWEEP SUMMARY =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")

    save_json(summary, os.path.join(OUT_DIR, "sweep_summary.json"))

    if RUN_TEST and TEST_JSON:
        print(f"\n===== Test predictions with alpha={best_alpha} =====")
        test_items = load_json(TEST_JSON)
        test_preds = predict_dataset(model, tokenizer, test_items, deltas, best_alpha)
        save_json(test_preds, os.path.join(OUT_DIR, "predictions_subtask1.json"))
        print(f"Saved {len(test_preds)} test predictions")


if __name__ == "__main__":
    main()

Device: cuda
Model: meta-llama/Llama-3.2-1B-Instruct
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Train: 769, Dev: 191
Total layers: 16, Steering layers: [12, 13, 14, 15]
Steering: 380 plausible + 380 implausible
Extracting plausible activations...
  100/380
  200/380
  300/380
Extracting implausible activations...
  100/380
  200/380
  300/380
  Layer 12: ||delta|| = 0.1564
  Layer 13: ||delta|| = 0.1961
  Layer 14: ||delta|| = 0.2374
  Layer 15: ||delta|| = 1.8168

===== Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :   0.00%  (0/49)
  invalid_plausible        :   0.00%  (0/46)
  valid_implausible        : 100.00%  (48/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_alpha_0.json
  ACC=50.26  TCE=50.0000  Score=10.1913

===== Alpha = -5 =====
  50/191
  100/191
  150/191
  invalid_implausible      :   0.00%  (0/49)
  invalid_plausible        :   0.00%  (0/46)
  valid_implausible        : 100.00%  (48/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_s

In [2]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

DEV_RATIO = 0.2
SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"
RUN_TEST = False
MODE = "kcast"

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    if USE_CHAT_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            if is_correct:
                correct_acts[l].append(hiddens[l])
            else:
                wrong_acts[l].append(hiddens[l])
            if gold:
                valid_acts[l].append(hiddens[l])
            else:
                invalid_acts[l].append(hiddens[l])

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    condition_valid = {}
    condition_invalid = {}
    for l in target_layers:
        condition_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        condition_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    return deltas, condition_valid, condition_invalid


def project_onto(v, u):
    dot = torch.dot(v, u)
    return dot / (torch.dot(u, u) + 1e-10) * u


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class KCASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid, layer_idx):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid
        self.layer_idx = layer_idx
        self.effective_alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        self.effective_alpha = effective_alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_static_hooks(layers, deltas, alpha):
    handles = []
    for layer_idx, delta in deltas.items():
        handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, alpha)))
    return handles


def register_kcast_hooks(layers, deltas, alpha, cond_valid, cond_invalid):
    handles = []
    for layer_idx, delta in deltas.items():
        hook = KCASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx], layer_idx)
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset_static(model, tokenizer, eval_items, deltas, alpha):
    layers = get_layers(model)
    handles = []
    if deltas is not None and alpha != 0:
        handles = register_static_hooks(layers, deltas, alpha)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_dataset_kcast(model, tokenizer, eval_items, deltas, alpha, cond_valid, cond_invalid):
    layers = get_layers(model)
    handles = register_kcast_hooks(layers, deltas, alpha, cond_valid, cond_invalid)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(dev_items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in dev_items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Alphas: {ALPHAS}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)
    save_json(train_items, os.path.join(OUT_DIR, "train_split.json"))
    save_json(dev_items, os.path.join(OUT_DIR, "dev_split.json"))
    print(f"Train: {len(train_items)}, Dev: {len(dev_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        if MODE == "kcast":
            if alpha == 0:
                preds = predict_dataset_static(model, tokenizer, dev_items, deltas, 0)
            else:
                preds = predict_dataset_kcast(model, tokenizer, dev_items, deltas, alpha, cond_valid, cond_invalid)
        else:
            preds = predict_dataset_static(model, tokenizer, dev_items, deltas, alpha)

        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(dev_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "dev_split.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))

    if RUN_TEST and TEST_JSON:
        print(f"\n===== Test predictions with alpha={best_alpha} =====")
        test_items = load_json(TEST_JSON)
        if MODE == "kcast":
            test_preds = predict_dataset_kcast(model, tokenizer, test_items, deltas, best_alpha, cond_valid, cond_invalid)
        else:
            test_preds = predict_dataset_static(model, tokenizer, test_items, deltas, best_alpha)
        save_json(test_preds, os.path.join(OUT_DIR, "predictions_subtask1.json"))
        print(f"Saved {len(test_preds)} test predictions")


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: kcast
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 769, Dev: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 758 examples
  100/758 (correct=59, wrong=41)
  200/758 (correct=122, wrong=78)
  300/758 (correct=179, wrong=121)
  400/758 (correct=247, wrong=153)
  500/758 (correct=318, wrong=182)
  600/758 (correct=381, wrong=219)
  700/758 (correct=448, wrong=252)
  Total correct=486, wrong=272
  Layer 21: ||delta|| = 3.1586
  Layer 22: ||delta|| = 4.1073
  Layer 23: ||delta|| = 4.6774
  Layer 24: ||delta|| = 5.3126
  Layer 25: ||delta|| = 6.2041
  Layer 26: ||delta|| = 6.9397
  Layer 27: ||delta|| = 6.8519

===== KCAST Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  61.22%  (30/49)
  invalid_plausible        :  10.87%  (5/46)
  valid_implausible        :  83.33%  (40/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_kcast_alpha_0.json
  ACC=64.40  TCE=44.5652  Score=13.3629

===== KCAST Alpha = -5 =====
  50/191

In [3]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

DEV_RATIO = 0.2
SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"
RUN_TEST = False
MODE = "kcast"

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    if USE_CHAT_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            if is_correct:
                correct_acts[l].append(hiddens[l])
            else:
                wrong_acts[l].append(hiddens[l])
            if gold:
                valid_acts[l].append(hiddens[l])
            else:
                invalid_acts[l].append(hiddens[l])

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    condition_valid = {}
    condition_invalid = {}
    for l in target_layers:
        condition_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        condition_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    return deltas, condition_valid, condition_invalid


def project_onto(v, u):
    dot = torch.dot(v, u)
    return dot / (torch.dot(u, u) + 1e-10) * u


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class KCASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid, layer_idx):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid
        self.layer_idx = layer_idx
        self.effective_alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        self.effective_alpha = effective_alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_static_hooks(layers, deltas, alpha):
    handles = []
    for layer_idx, delta in deltas.items():
        handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, alpha)))
    return handles


def register_kcast_hooks(layers, deltas, alpha, cond_valid, cond_invalid):
    handles = []
    for layer_idx, delta in deltas.items():
        hook = KCASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx], layer_idx)
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset_static(model, tokenizer, eval_items, deltas, alpha):
    layers = get_layers(model)
    handles = []
    if deltas is not None and alpha != 0:
        handles = register_static_hooks(layers, deltas, alpha)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_dataset_kcast(model, tokenizer, eval_items, deltas, alpha, cond_valid, cond_invalid):
    layers = get_layers(model)
    handles = register_kcast_hooks(layers, deltas, alpha, cond_valid, cond_invalid)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(dev_items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in dev_items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Alphas: {ALPHAS}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)
    save_json(train_items, os.path.join(OUT_DIR, "train_split.json"))
    save_json(dev_items, os.path.join(OUT_DIR, "dev_split.json"))
    print(f"Train: {len(train_items)}, Dev: {len(dev_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        if MODE == "kcast":
            if alpha == 0:
                preds = predict_dataset_static(model, tokenizer, dev_items, deltas, 0)
            else:
                preds = predict_dataset_kcast(model, tokenizer, dev_items, deltas, alpha, cond_valid, cond_invalid)
        else:
            preds = predict_dataset_static(model, tokenizer, dev_items, deltas, alpha)

        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(dev_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "dev_split.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))

    if RUN_TEST and TEST_JSON:
        print(f"\n===== Test predictions with alpha={best_alpha} =====")
        test_items = load_json(TEST_JSON)
        if MODE == "kcast":
            test_preds = predict_dataset_kcast(model, tokenizer, test_items, deltas, best_alpha, cond_valid, cond_invalid)
        else:
            test_preds = predict_dataset_static(model, tokenizer, test_items, deltas, best_alpha)
        save_json(test_preds, os.path.join(OUT_DIR, "predictions_subtask1.json"))
        print(f"Saved {len(test_preds)} test predictions")


if __name__ == "__main__":
    main()

Device: cuda
Model: meta-llama/Llama-3.2-1B-Instruct
Mode: kcast
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Train: 769, Dev: 191
Total layers: 16, Steering layers: [12, 13, 14, 15]
Steering pool: 758 examples
  100/758 (correct=44, wrong=56)
  200/758 (correct=97, wrong=103)
  300/758 (correct=145, wrong=155)
  400/758 (correct=194, wrong=206)
  500/758 (correct=250, wrong=250)
  600/758 (correct=304, wrong=296)
  700/758 (correct=353, wrong=347)
  Total correct=380, wrong=378
  Layer 12: ||delta|| = 0.3230
  Layer 13: ||delta|| = 0.3981
  Layer 14: ||delta|| = 0.4877
  Layer 15: ||delta|| = 3.7656

===== KCAST Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :   0.00%  (0/49)
  invalid_plausible        :   0.00%  (0/46)
  valid_implausible        : 100.00%  (48/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_kcast_alpha_0.json
  ACC=50.26  TCE=50.0000  Score=10.1913

===== KCAST Alpha = -5 =====
  50/191
  100/191
  150/191
  invalid_implausible      :   0.00%  (0/49)
  invalid_plausible        :   0.00%  (0

In [4]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

DEV_RATIO = 0.2
SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"
RUN_TEST = False

# "static" = fixed alpha for all inputs
# "cast"   = conditional: flip alpha sign based on valid/invalid similarity (paper eq 3)
# "kcast"  = k-nearest neighbor weighted alpha + conditional sign flip
MODE = "kcast"

# kcast specific
KCAST_K = 10
KCAST_TEMPERATURE = 0.0

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    if USE_CHAT_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_labels = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
            all_acts_with_labels[l].append((h, is_correct))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_labels[l]])
        labels = torch.tensor([1.0 if item[1] else 0.0 for item in all_acts_with_labels[l]])
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return deltas, cond_valid, cond_invalid, knn_store


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid, knn_vecs, knn_labels, k, temperature):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k
        self.temperature = temperature

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()

        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            sign = -1.0
        else:
            sign = 1.0

        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        topk_sims = topk.values
        topk_labels = self.knn_labels[topk.indices]

        weights = torch.softmax(topk_sims / self.temperature, dim=0)
        correctness_score = (weights * topk_labels).sum().item()

        magnitude = self.alpha * (1.0 - correctness_score)
        effective_alpha = sign * magnitude

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    handles = []
    for layer_idx, delta in deltas.items():
        if mode == "static":
            hook = StaticHook(delta, alpha)
        elif mode == "cast":
            hook = CASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(
                delta, alpha,
                cond_valid[layer_idx], cond_invalid[layer_idx],
                knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"],
                KCAST_K, KCAST_TEMPERATURE
            )
        else:
            raise ValueError(f"Unknown mode: {mode}")
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset(model, tokenizer, eval_items, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    layers = get_layers(model)
    handles = []
    if alpha != 0:
        handles = register_hooks(layers, deltas, alpha, mode, cond_valid, cond_invalid, knn_store)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(dev_items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in dev_items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}, Temperature={KCAST_TEMPERATURE}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)
    save_json(train_items, os.path.join(OUT_DIR, "train_split.json"))
    save_json(dev_items, os.path.join(OUT_DIR, "dev_split.json"))
    print(f"Train: {len(train_items)}, Dev: {len(dev_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid, knn_store = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid, "knn_store": knn_store}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, dev_items, deltas, alpha, MODE, cond_valid, cond_invalid, knn_store)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(dev_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "dev_split.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))

    if RUN_TEST and TEST_JSON:
        print(f"\n===== Test predictions with alpha={best_alpha} =====")
        test_items = load_json(TEST_JSON)
        test_preds = predict_dataset(model, tokenizer, test_items, deltas, best_alpha, MODE, cond_valid, cond_invalid, knn_store)
        save_json(test_preds, os.path.join(OUT_DIR, "predictions_subtask1.json"))
        print(f"Saved {len(test_preds)} test predictions")


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: kcast
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
K-CAST K=10, Temperature=1.0


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 769, Dev: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 758 examples
  100/758 (correct=59, wrong=41)
  200/758 (correct=122, wrong=78)
  300/758 (correct=179, wrong=121)
  400/758 (correct=247, wrong=153)
  500/758 (correct=318, wrong=182)
  600/758 (correct=381, wrong=219)
  700/758 (correct=448, wrong=252)
  Total correct=486, wrong=272
  Layer 21: ||delta|| = 3.1586
  Layer 22: ||delta|| = 4.1073
  Layer 23: ||delta|| = 4.6774
  Layer 24: ||delta|| = 5.3126
  Layer 25: ||delta|| = 6.2041
  Layer 26: ||delta|| = 6.9397
  Layer 27: ||delta|| = 6.8519

===== KCAST Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  61.22%  (30/49)
  invalid_plausible        :  10.87%  (5/46)
  valid_implausible        :  83.33%  (40/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_kcast_alpha_0.json
  ACC=64.40  TCE=44.5652  Score=13.3629

===== KCAST Alpha = -5 =====
  50/191

In [7]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

DEV_RATIO = 0.2
SEED = 1111
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"
RUN_TEST = True

# "static" = fixed alpha
# "cast"   = sign flip via projection similarity (eq 3)
# "kcast"  = sign flip via kNN majority vote (eq 4-5)
MODE = "static"

KCAST_K = 10

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (it["validity"], it["plausibility"])
        buckets.setdefault(key, []).append(it)
    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        n_dev = max(1, int(len(bucket) * dev_ratio)) if len(bucket) > 5 else max(0, int(len(bucket) * dev_ratio))
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])
    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    if USE_CHAT_TEMPLATE:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return deltas, cond_valid, cond_invalid, knn_store


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()

        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()

        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    handles = []
    for layer_idx, delta in deltas.items():
        if mode == "static":
            hook = StaticHook(delta, alpha)
        elif mode == "cast":
            hook = CASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(delta, alpha, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        else:
            raise ValueError(f"Unknown mode: {mode}")
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset(model, tokenizer, eval_items, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    layers = get_layers(model)
    handles = []
    if alpha != 0:
        handles = register_hooks(layers, deltas, alpha, mode, cond_valid, cond_invalid, knn_store)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(dev_items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in dev_items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    items = load_json(TRAIN_JSON)
    train_items, dev_items = stratified_split(items, DEV_RATIO, SEED)
    save_json(train_items, os.path.join(OUT_DIR, "train_split.json"))
    save_json(dev_items, os.path.join(OUT_DIR, "dev_split.json"))
    print(f"Train: {len(train_items)}, Dev: {len(dev_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid, knn_store = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid, "knn_store": knn_store}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, dev_items, deltas, alpha, MODE, cond_valid, cond_invalid, knn_store)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(dev_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "dev_split.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))

    if RUN_TEST and TEST_JSON:
        print(f"\n===== Test predictions with alpha={best_alpha} =====")
        test_items = load_json(TEST_JSON)
        test_preds = predict_dataset(model, tokenizer, test_items, deltas, best_alpha, MODE, cond_valid, cond_invalid, knn_store)
        save_json(test_preds, os.path.join(OUT_DIR, "predictions_subtask1.json"))
        print(f"Saved {len(test_preds)} test predictions")


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: static
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 769, Dev: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 758 examples
  100/758 (correct=66, wrong=34)
  200/758 (correct=128, wrong=72)
  300/758 (correct=194, wrong=106)
  400/758 (correct=250, wrong=150)
  500/758 (correct=318, wrong=182)
  600/758 (correct=374, wrong=226)
  700/758 (correct=446, wrong=254)
  Total correct=483, wrong=275
  Layer 21: ||delta|| = 3.1828
  Layer 22: ||delta|| = 3.9458
  Layer 23: ||delta|| = 4.6202
  Layer 24: ||delta|| = 5.3025
  Layer 25: ||delta|| = 6.3142
  Layer 26: ||delta|| = 7.1571
  Layer 27: ||delta|| = 6.9741

===== STATIC Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  65.31%  (32/49)
  invalid_plausible        :  15.22%  (7/46)
  valid_implausible        :  87.50%  (42/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_static_alpha_0.json
  ACC=67.54  TCE=42.3913  Score=14.1584

===== STATIC Alpha = -5 =====
  50/

In [8]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

SEED = 1
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = True
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

# "static" / "cast" / "kcast"
MODE = "kcast"
KCAST_K = 10

# "zero_shot" / "icl"
PROMPT_MODE = "zero_shot"
ICL_SHOTS = 4

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return deltas, cond_valid, cond_invalid, knn_store


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    handles = []
    for layer_idx, delta in deltas.items():
        if mode == "static":
            hook = StaticHook(delta, alpha)
        elif mode == "cast":
            hook = CASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(delta, alpha, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        else:
            raise ValueError(f"Unknown mode: {mode}")
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    layers = get_layers(model)
    handles = []
    if alpha != 0:
        handles = register_hooks(layers, deltas, alpha, mode, cond_valid, cond_invalid, knn_store)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid, knn_store = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid, "knn_store": knn_store}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, deltas, alpha, MODE, cond_valid, cond_invalid, knn_store)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "test_reference.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: kcast
Prompt: zero_shot
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
K-CAST K=10


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=64, wrong=36)
  200/945 (correct=126, wrong=74)
  300/945 (correct=190, wrong=110)
  400/945 (correct=253, wrong=147)
  500/945 (correct=320, wrong=180)
  600/945 (correct=383, wrong=217)
  700/945 (correct=442, wrong=258)
  800/945 (correct=505, wrong=295)
  900/945 (correct=575, wrong=325)
  Total correct=607, wrong=338
  Layer 21: ||delta|| = 1.0000
  Layer 22: ||delta|| = 1.0000
  Layer 23: ||delta|| = 1.0000
  Layer 24: ||delta|| = 1.0000
  Layer 25: ||delta|| = 1.0000
  Layer 26: ||delta|| = 1.0000
  Layer 27: ||delta|| = 1.0000

===== KCAST Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  70.83%  (34/48)
  invalid_plausible        :  19.15%  (9/47)
  valid_implausible        :  83.33%  (40/48)
  valid_plausible          :  97.92%  (47/48)
Scoring complete. Results written to results_steering/eval_kcast_alpha_0.json
  ACC=6

In [12]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

SEED = 1
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

# "static" / "cast" / "kcast"
MODE = "kcast"
KCAST_K = 10

# "zero_shot" / "icl"
PROMPT_MODE = "icl"
ICL_SHOTS = 4

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return deltas, cond_valid, cond_invalid, knn_store


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    handles = []
    for layer_idx, delta in deltas.items():
        if mode == "static":
            hook = StaticHook(delta, alpha)
        elif mode == "cast":
            hook = CASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(delta, alpha, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        else:
            raise ValueError(f"Unknown mode: {mode}")
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    layers = get_layers(model)
    handles = []
    if alpha != 0:
        handles = register_hooks(layers, deltas, alpha, mode, cond_valid, cond_invalid, knn_store)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid, knn_store = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid, "knn_store": knn_store}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, deltas, alpha, MODE, cond_valid, cond_invalid, knn_store)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "test_reference.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: kcast
Prompt: icl (4 shots)
Alphas: [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
K-CAST K=10


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=65, wrong=35)
  200/945 (correct=120, wrong=80)
  300/945 (correct=185, wrong=115)
  400/945 (correct=244, wrong=156)
  500/945 (correct=310, wrong=190)
  600/945 (correct=369, wrong=231)
  700/945 (correct=426, wrong=274)
  800/945 (correct=490, wrong=310)
  900/945 (correct=553, wrong=347)
  Total correct=585, wrong=360
  Layer 21: ||delta|| = 3.6602
  Layer 22: ||delta|| = 4.9387
  Layer 23: ||delta|| = 5.5657
  Layer 24: ||delta|| = 6.5616
  Layer 25: ||delta|| = 7.7881
  Layer 26: ||delta|| = 9.3934
  Layer 27: ||delta|| = 8.2923

===== KCAST Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  60.42%  (29/48)
  invalid_plausible        :  17.02%  (8/47)
  valid_implausible        :  93.75%  (45/48)
  valid_plausible          : 100.00%  (48/48)
Scoring complete. Results written to results_steering/eval_kcast_alpha_0.json
  ACC=6

KeyboardInterrupt: 

In [10]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_steering"

SEED = 1
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

# "static" / "cast" / "kcast"
MODE = "static"
KCAST_K = 10

# "zero_shot" / "icl"
PROMPT_MODE = "icl"
ICL_SHOTS = 4

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


def compute_steering_data(model, tokenizer, train_items, target_layers):
    plausible = [x for x in train_items if x["plausibility"] is True]
    implausible = [x for x in train_items if x["plausibility"] is False]
    n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
    pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
    random.Random(SEED).shuffle(pool)
    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return deltas, cond_valid, cond_invalid, knn_store


class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()

        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha

        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)

        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    handles = []
    for layer_idx, delta in deltas.items():
        if mode == "static":
            hook = StaticHook(delta, alpha)
        elif mode == "cast":
            hook = CASTHook(delta, alpha, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(delta, alpha, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        else:
            raise ValueError(f"Unknown mode: {mode}")
        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, deltas, alpha, mode, cond_valid=None, cond_invalid=None, knn_store=None):
    layers = get_layers(model)
    handles = []
    if alpha != 0:
        handles = register_hooks(layers, deltas, alpha, mode, cond_valid, cond_invalid, knn_store)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    deltas, cond_valid, cond_invalid, knn_store = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save({"deltas": deltas, "cond_valid": cond_valid, "cond_invalid": cond_invalid, "knn_store": knn_store}, os.path.join(OUT_DIR, "steering_data.pt"))

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, deltas, alpha, MODE, cond_valid, cond_invalid, knn_store)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, os.path.join(OUT_DIR, "test_reference.json"), pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: meta-llama/Llama-3.2-1B-Instruct
Mode: static
Prompt: icl (4 shots)
Alphas: [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Train: 960, Test: 191
Total layers: 16, Steering layers: [12, 13, 14, 15]
Steering pool: 945 examples
  100/945 (correct=60, wrong=40)
  200/945 (correct=113, wrong=87)
  300/945 (correct=174, wrong=126)
  400/945 (correct=226, wrong=174)
  500/945 (correct=291, wrong=209)
  600/945 (correct=339, wrong=261)
  700/945 (correct=391, wrong=309)
  800/945 (correct=448, wrong=352)
  900/945 (correct=496, wrong=404)
  Total correct=522, wrong=423
  Layer 12: ||delta|| = 0.2001
  Layer 13: ||delta|| = 0.2563
  Layer 14: ||delta|| = 0.3048
  Layer 15: ||delta|| = 1.5562

===== STATIC Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  35.42%  (17/48)
  invalid_plausible        :   6.38%  (3/47)
  valid_implausible        :  81.25%  (39/48)
  valid_plausible          :  93.75%  (45/48)
Scoring complete. Results written to results_steering/eval_static_alpha_0.json
  ACC=54.45  TCE=43.6835  Score=11.3447

===== STATIC Alpha = -3 =====
  50/191
  100/191
  150/191
  invalid_

In [2]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "icl"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "entropy_gated"

KCAST_K = 14
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: layerwise
Prompt: zero_shot
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=67, wrong=33)
  200/945 (correct=131, wrong=69)
  300/945 (correct=199, wrong=101)
  400/945 (correct=268, wrong=132)
  500/945 (correct=328, wrong=172)
  600/945 (correct=385, wrong=215)
  700/945 (correct=453, wrong=247)
  800/945 (correct=517, wrong=283)
  900/945 (correct=578, wrong=322)
  Total correct=608, wrong=337
  Layer 21: ||delta|| = 3.1608
  Layer 22: ||delta|| = 4.0241
  Layer 23: ||delta|| = 4.6366
  Layer 24: ||delta|| = 5.2824
  Layer 25: ||delta|| = 6.1881
  Layer 26: ||delta|| = 6.9601
  Layer 27: ||delta|| = 6.8585
  Layer 21: ||valid_delta|| = 17.8768, ||invalid_delta|| = 18.7794
  Layer 22: ||valid_delta|| = 26.2211, ||invalid_delta|| = 27.9916
  Layer 23: ||valid_delta|| = 26.9883, ||invalid_delta|| = 28.9778
  Layer 24: ||valid_delta|| = 29.6446, ||invalid_delta|| = 31.8733
  Layer 25: ||valid_delta|| = 35.0693, ||invalid_delta|| =

KeyboardInterrupt: 

In [1]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "icl"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "entropy_gated"

KCAST_K = 14
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-3B-Instruct
Mode: entropy_gated
Prompt: icl (4 shots)
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
Entropy threshold: 0.5


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Train: 960, Test: 191
Total layers: 36, Steering layers: [27, 28, 29, 30, 31, 32, 33, 34, 35]
Steering pool: 945 examples
  100/945 (correct=74, wrong=26)
  200/945 (correct=153, wrong=47)
  300/945 (correct=216, wrong=84)
  400/945 (correct=288, wrong=112)
  500/945 (correct=366, wrong=134)
  600/945 (correct=434, wrong=166)
  700/945 (correct=497, wrong=203)
  800/945 (correct=572, wrong=228)
  900/945 (correct=642, wrong=258)
  Total correct=672, wrong=273
  Layer 27: ||delta|| = 4.6097
  Layer 28: ||delta|| = 6.2293
  Layer 29: ||delta|| = 6.5263
  Layer 30: ||delta|| = 7.5304
  Layer 31: ||delta|| = 8.7289
  Layer 32: ||delta|| = 10.1700
  Layer 33: ||delta|| = 11.6670
  Layer 34: ||delta|| = 13.9806
  Layer 35: ||delta|| = 11.8814
  Layer 27: ||valid_delta|| = 25.2179, ||invalid_delta|| = 29.7007
  Layer 28: ||valid_delta|| = 32.1684, ||invalid_delta|| = 37.7717
  Layer 29: ||valid_delta|| = 34.2957, ||invalid_delta|| = 39.5369
  Layer 30: ||valid_delta|| = 44.1864, ||invalid_del

In [3]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "zero_shot"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "structure_aware"

KCAST_K = 10
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: structure_aware
Prompt: zero_shot
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Structure-aware: 318 paired, 324 unpaired
Steering pool: 960 examples
  100/960 (correct=60, wrong=40)
  200/960 (correct=123, wrong=77)
  300/960 (correct=184, wrong=116)
  400/960 (correct=250, wrong=150)
  500/960 (correct=314, wrong=186)
  600/960 (correct=380, wrong=220)
  700/960 (correct=447, wrong=253)
  800/960 (correct=517, wrong=283)
  900/960 (correct=580, wrong=320)
  Total correct=619, wrong=341
  Layer 21: ||delta|| = 3.2021
  Layer 22: ||delta|| = 4.0875
  Layer 23: ||delta|| = 4.7055
  Layer 24: ||delta|| = 5.3585
  Layer 25: ||delta|| = 6.2773
  Layer 26: ||delta|| = 7.0571
  Layer 27: ||delta|| = 6.9502
  Layer 21: ||valid_delta|| = 17.9189, ||invalid_delta|| = 18.7578
  Layer 22: ||valid_delta|| = 26.3100, ||invalid_delta|| = 27.9739
  Layer 23: ||valid_delta|| = 27.0804, ||invalid_delta|| = 28.9647
  Layer 24: ||valid_delta|| = 29.7421, ||invalid_delta|| = 31.8599
  Layer 25: ||va

KeyboardInterrupt: 

In [5]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]
LAYER_FRACTION = 0.125
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "zero_shot"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "static"

KCAST_K = 10
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct
Mode: static
Prompt: zero_shot
Alphas: [0, -5, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 5, 7, 10]


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Train: 960, Test: 191
Total layers: 28, Steering layers: [24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=67, wrong=33)
  200/945 (correct=131, wrong=69)
  300/945 (correct=199, wrong=101)
  400/945 (correct=268, wrong=132)
  500/945 (correct=328, wrong=172)
  600/945 (correct=385, wrong=215)
  700/945 (correct=453, wrong=247)
  800/945 (correct=517, wrong=283)
  900/945 (correct=578, wrong=322)
  Total correct=608, wrong=337
  Layer 24: ||delta|| = 5.2824
  Layer 25: ||delta|| = 6.1881
  Layer 26: ||delta|| = 6.9601
  Layer 27: ||delta|| = 6.8585
  Layer 24: ||valid_delta|| = 29.6446, ||invalid_delta|| = 31.8733
  Layer 25: ||valid_delta|| = 35.0693, ||invalid_delta|| = 36.8521
  Layer 26: ||valid_delta|| = 35.7527, ||invalid_delta|| = 38.3718
  Layer 27: ||valid_delta|| = 34.1142, ||invalid_delta|| = 37.7296

===== STATIC Alpha = 0 =====
  50/191
  100/191
  150/191
  invalid_implausible      :  70.83%  (34/48)
  invalid_plausible        :  19.15%  (9/47)
  valid_impla

In [4]:
import os

os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/cache/huggingface"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/workspace/cache/huggingface"
os.environ["PIP_CACHE_DIR"] = "/workspace/cache/pip"
os.environ["TMPDIR"] = "/workspace/cache/tmp"


In [2]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0,  -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "icl"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "static"

KCAST_K = 10
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-3B-Instruct
Mode: static
Prompt: icl (4 shots)
Alphas: [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Train: 960, Test: 191
Total layers: 36, Steering layers: [27, 28, 29, 30, 31, 32, 33, 34, 35]
Steering pool: 945 examples
  100/945 (correct=73, wrong=27)
  200/945 (correct=151, wrong=49)
  300/945 (correct=214, wrong=86)
  400/945 (correct=287, wrong=113)
  500/945 (correct=365, wrong=135)
  600/945 (correct=434, wrong=166)
  700/945 (correct=498, wrong=202)
  800/945 (correct=575, wrong=225)
  900/945 (correct=645, wrong=255)
  Total correct=676, wrong=269
  Layer 27: ||delta|| = 4.3477
  Layer 28: ||delta|| = 5.8903
  Layer 29: ||delta|| = 6.1787
  Layer 30: ||delta|| = 7.1254
  Layer 31: ||delta|| = 8.2568
  Layer 32: ||delta|| = 9.6155
  Layer 33: ||delta|| = 11.0195
  Layer 34: ||delta|| = 13.1428
  Layer 35: ||delta|| = 11.1253
  Layer 27: ||valid_delta|| = 25.3720, ||invalid_delta|| = 29.4662
  Layer 28: ||valid_delta|| = 32.2588, ||invalid_delta|| = 37.4614
  Layer 29: ||valid_delta|| = 34.3803, ||invalid_delta|| = 39.2461
  Layer 30: ||valid_delta|| = 44.2321, ||invalid_delt

In [2]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0,  -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "icl"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "static"

KCAST_K = 10
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-7B-Instruct
Mode: static
Prompt: icl (4 shots)
Alphas: [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=72, wrong=28)
  200/945 (correct=146, wrong=54)
  300/945 (correct=215, wrong=85)
  400/945 (correct=286, wrong=114)
  500/945 (correct=360, wrong=140)
  600/945 (correct=439, wrong=161)
  700/945 (correct=499, wrong=201)
  800/945 (correct=572, wrong=228)
  900/945 (correct=647, wrong=253)
  Total correct=680, wrong=265
  Layer 21: ||delta|| = 9.3381
  Layer 22: ||delta|| = 11.6734
  Layer 23: ||delta|| = 13.4626
  Layer 24: ||delta|| = 16.7353
  Layer 25: ||delta|| = 20.8493
  Layer 26: ||delta|| = 29.1729
  Layer 27: ||delta|| = 41.5705
  Layer 21: ||valid_delta|| = 45.3653, ||invalid_delta|| = 45.5164
  Layer 22: ||valid_delta|| = 67.1004, ||invalid_delta|| = 68.3424
  Layer 23: ||valid_delta|| = 100.0145, ||invalid_delta|| = 101.6027
  Layer 24: ||valid_delta|| = 112.8569, ||invalid_delta|| = 116.1865
  Layer 25: ||valid_delta|| = 125.2589, ||invalid

KeyboardInterrupt: 

In [1]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [0,  -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "icl"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "kcast"

KCAST_K = 10
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-7B-Instruct
Mode: kcast
Prompt: icl (4 shots)
Alphas: [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]
K-CAST K=10


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=72, wrong=28)
  200/945 (correct=146, wrong=54)
  300/945 (correct=215, wrong=85)
  400/945 (correct=286, wrong=114)
  500/945 (correct=360, wrong=140)
  600/945 (correct=439, wrong=161)
  700/945 (correct=499, wrong=201)
  800/945 (correct=572, wrong=228)
  900/945 (correct=647, wrong=253)
  Total correct=680, wrong=265
  Layer 21: ||delta|| = 9.3381
  Layer 22: ||delta|| = 11.6734
  Layer 23: ||delta|| = 13.4626
  Layer 24: ||delta|| = 16.7353
  Layer 25: ||delta|| = 20.8493
  Layer 26: ||delta|| = 29.1729
  Layer 27: ||delta|| = 41.5705
  Layer 21: ||valid_delta|| = 45.3653, ||invalid_delta|| = 45.5164
  Layer 22: ||valid_delta|| = 67.1004, ||invalid_delta|| = 68.3424
  Layer 23: ||valid_delta|| = 100.0145, ||invalid_delta|| = 101.6027
  Layer 24: ||valid_delta|| = 112.8569, ||invalid_delta|| = 116.1865
  Layer 25: ||valid_delta|| = 125.2589, ||invalid

In [1]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 1111
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [-4, -3.5, -3, -2.5, -2]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "icl"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "kcast"

KCAST_K = 10
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-7B-Instruct
Mode: kcast
Prompt: icl (4 shots)
Alphas: [-4, -3.5, -3, -2.5, -2]
K-CAST K=10


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=75, wrong=25)
  200/945 (correct=145, wrong=55)
  300/945 (correct=224, wrong=76)
  400/945 (correct=301, wrong=99)
  500/945 (correct=374, wrong=126)
  600/945 (correct=444, wrong=156)
  700/945 (correct=518, wrong=182)
  800/945 (correct=596, wrong=204)
  900/945 (correct=666, wrong=234)
  Total correct=692, wrong=253
  Layer 21: ||delta|| = 10.7316
  Layer 22: ||delta|| = 12.9791
  Layer 23: ||delta|| = 15.1634
  Layer 24: ||delta|| = 18.4848
  Layer 25: ||delta|| = 23.3917
  Layer 26: ||delta|| = 31.4891
  Layer 27: ||delta|| = 40.5526
  Layer 21: ||valid_delta|| = 45.9948, ||invalid_delta|| = 39.2380
  Layer 22: ||valid_delta|| = 58.0265, ||invalid_delta|| = 51.2659
  Layer 23: ||valid_delta|| = 81.0590, ||invalid_delta|| = 70.5247
  Layer 24: ||valid_delta|| = 93.1420, ||invalid_delta|| = 82.2790
  Layer 25: ||valid_delta|| = 105.9974, ||invalid_del

In [3]:
import os
import json
import random
import importlib.util
from typing import List, Dict, Any, Tuple
from itertools import product

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HOME = os.getenv("HF_HOME", "")
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUT_DIR = "results_novelty"

SEED = 42
MAX_LEN = 512
MAX_STEER_EXAMPLES = 2400
ALPHAS = [-4, -3.5, -3, -2.5, -2]
LAYER_FRACTION = 0.25
NORMALIZE_VECTORS = False
USE_BFLOAT16 = True
USE_CHAT_TEMPLATE = True
YES_TEXT = " yes"
NO_TEXT = " no"

PROMPT_MODE = "icl"
ICL_SHOTS = 4

# "static" / "cast" / "kcast"
# "layerwise"          = novelty 1: per-layer alpha optimization
# "validity_cond"      = novelty 2: separate vectors for valid vs invalid
# "entropy_gated"      = novelty 3: entropy-based alpha scaling
# "structure_aware"    = novelty 5: syllogistic form aware contrastive pairs
# "layerwise_entropy"  = novelty 1+3 combined
MODE = "kcast"

KCAST_K = 10
ENTROPY_THRESHOLD = 0.5
LAYERWISE_ALPHA_VALUES = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
LAYERWISE_MAX_COMBOS = 5000

if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "module"):
        return get_layers(model.module)
    raise ValueError("Unsupported model architecture")


def get_target_layers(total, fraction):
    start = int(total * (1.0 - fraction))
    return list(range(start, total))


def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    if PROMPT_MODE == "icl":
        chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
        if USE_CHAT_TEMPLATE:
            messages = [
                {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."}
            ]
            for ex in chosen:
                ans = "yes" if ex["validity"] else "no"
                messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
                messages.append({"role": "assistant", "content": ans})
            messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        parts = [
            "You are a strict formal logic reasoner.",
            "Decide only whether the conclusion logically follows from the premises.",
            "Ignore plausibility and world knowledge.",
            "Reply with only yes or no.", ""
        ]
        for ex in chosen:
            ans = "yes" if ex["validity"] else "no"
            parts.extend([f"Argument:\n{ex['syllogism']}", f"Answer: {ans}", ""])
        parts.extend([f"Argument:\n{syllogism}", "Answer:"])
        return "\n".join(parts)

    if USE_CHAT_TEMPLATE:
        messages = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        "You are a strict formal logic reasoner.\n"
        "Decide only whether the conclusion logically follows from the premises.\n"
        "Ignore plausibility and world knowledge.\n"
        "Reply with only yes or no.\n\n"
        f"Argument:\n{syllogism}\n\nAnswer:"
    )


def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        out = model(full_ids, use_cache=False)
        logits = out.logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    start = max(prompt_ids.shape[1] - 1, 0)
    return lp[0, start:].sum().item()


def predict_validity(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    return y >= n


def get_prediction_entropy(model, tokenizer, prompt, max_len):
    y = score_label(model, tokenizer, prompt, YES_TEXT, max_len)
    n = score_label(model, tokenizer, prompt, NO_TEXT, max_len)
    logits = torch.tensor([y, n])
    probs = torch.softmax(logits, dim=0)
    entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
    max_entropy = np.log(2)
    return entropy / max_entropy, y >= n


def get_all_layer_hidden(model, tokenizer, text, target_layers, max_len):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=False)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    return {l: out.hidden_states[l + 1][0, -1, :].float().cpu() for l in target_layers}


def balanced_by_validity(data, n, seed):
    rng = random.Random(seed)
    pos = [x for x in data if x["validity"] is True]
    neg = [x for x in data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = n // 2
    out = pos[:half] + neg[:n - half]
    rng.shuffle(out)
    return out


def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-10)


def project_onto(v, u):
    return (torch.dot(v, u) / (torch.dot(u, u) + 1e-10)) * u


# ============================================================
# NOVELTY 5: Structure-aware contrastive pairs
# ============================================================

def extract_syllogistic_form(text):
    text_lower = text.lower()
    quantifiers = ["not all", "no", "all", "some", "every", "there are no", "it is not the case that all",
                   "it is not the case that every", "there exist some", "every single", "it is certain that no",
                   "it is certain that every", "it is also true that every", "it is known that some",
                   "it is known that every", "it is known that no"]
    found = []
    for q in sorted(quantifiers, key=len, reverse=True):
        if q in text_lower:
            found.append(q)
    if "therefore" in text_lower or "consequently" in text_lower or "it follows" in text_lower or "this has led" in text_lower:
        found.append("CONCLUSION_MARKER")
    neg_count = text_lower.count("not") + text_lower.count("no ")
    form_key = "|".join(sorted(found)) + f"|neg{neg_count}"
    return form_key


def build_structure_aware_pool(train_items, max_examples, seed):
    from collections import defaultdict
    rng = random.Random(seed)

    groups = defaultdict(lambda: {"valid_plausible": [], "valid_implausible": [],
                                   "invalid_plausible": [], "invalid_implausible": []})
    for item in train_items:
        form = extract_syllogistic_form(item["syllogism"])
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item["plausibility"] else "implausible"
        groups[form][f"{v}_{p}"].append(item)

    paired = []
    unpaired = []
    for form, buckets in groups.items():
        vp = buckets["valid_plausible"]
        vi = buckets["valid_implausible"]
        ip = buckets["invalid_plausible"]
        ii = buckets["invalid_implausible"]
        rng.shuffle(vp)
        rng.shuffle(vi)
        rng.shuffle(ip)
        rng.shuffle(ii)
        n_valid = min(len(vp), len(vi))
        for j in range(n_valid):
            paired.append((vp[j], vi[j]))
        n_invalid = min(len(ip), len(ii))
        for j in range(n_invalid):
            paired.append((ip[j], ii[j]))
        for lst in [vp[n_valid:], vi[n_valid:], ip[n_invalid:], ii[n_invalid:]]:
            unpaired.extend(lst)

    rng.shuffle(paired)
    rng.shuffle(unpaired)
    print(f"Structure-aware: {len(paired)} paired, {len(unpaired)} unpaired")

    pool = []
    for p_item, ip_item in paired[:max_examples // 2]:
        pool.append(p_item)
        pool.append(ip_item)
    remaining = max_examples - len(pool)
    if remaining > 0:
        pool.extend(unpaired[:remaining])
    rng.shuffle(pool)
    return pool


# ============================================================
# Core steering data computation
# ============================================================

def compute_steering_data(model, tokenizer, train_items, target_layers):
    if MODE == "structure_aware":
        pool = build_structure_aware_pool(train_items, MAX_STEER_EXAMPLES, SEED)
    else:
        plausible = [x for x in train_items if x["plausibility"] is True]
        implausible = [x for x in train_items if x["plausibility"] is False]
        n = min(len(plausible), len(implausible), MAX_STEER_EXAMPLES // 2)
        pool = balanced_by_validity(plausible, n, SEED) + balanced_by_validity(implausible, n, SEED)
        random.Random(SEED).shuffle(pool)

    print(f"Steering pool: {len(pool)} examples")

    correct_acts = {l: [] for l in target_layers}
    wrong_acts = {l: [] for l in target_layers}
    valid_acts = {l: [] for l in target_layers}
    invalid_acts = {l: [] for l in target_layers}
    all_acts_with_validity = {l: [] for l in target_layers}

    valid_correct_acts = {l: [] for l in target_layers}
    valid_wrong_acts = {l: [] for l in target_layers}
    invalid_correct_acts = {l: [] for l in target_layers}
    invalid_wrong_acts = {l: [] for l in target_layers}

    n_correct = 0
    n_wrong = 0

    for i, ex in enumerate(pool):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        hiddens = get_all_layer_hidden(model, tokenizer, prompt, target_layers, MAX_LEN)
        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        gold = bool(ex["validity"])
        is_correct = (pred == gold)

        if is_correct:
            n_correct += 1
        else:
            n_wrong += 1

        for l in target_layers:
            h = hiddens[l]
            if is_correct:
                correct_acts[l].append(h)
            else:
                wrong_acts[l].append(h)
            if gold:
                valid_acts[l].append(h)
                if is_correct:
                    valid_correct_acts[l].append(h)
                else:
                    valid_wrong_acts[l].append(h)
            else:
                invalid_acts[l].append(h)
                if is_correct:
                    invalid_correct_acts[l].append(h)
                else:
                    invalid_wrong_acts[l].append(h)
            all_acts_with_validity[l].append((h, 1 if gold else -1))

        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(pool)} (correct={n_correct}, wrong={n_wrong})")

    print(f"  Total correct={n_correct}, wrong={n_wrong}")

    deltas = {}
    for l in target_layers:
        if len(correct_acts[l]) == 0 or len(wrong_acts[l]) == 0:
            raise ValueError(f"Layer {l}: need both correct and wrong examples")
        mu_plus = torch.stack(correct_acts[l]).mean(dim=0)
        mu_minus = torch.stack(wrong_acts[l]).mean(dim=0)
        delta = mu_plus - mu_minus
        if NORMALIZE_VECTORS:
            delta = delta / delta.norm()
        deltas[l] = delta
        print(f"  Layer {l}: ||delta|| = {delta.norm():.4f}")

    valid_deltas = {}
    invalid_deltas = {}
    for l in target_layers:
        if len(valid_correct_acts[l]) > 0 and len(valid_wrong_acts[l]) > 0:
            vd = torch.stack(valid_correct_acts[l]).mean(dim=0) - torch.stack(valid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                vd = vd / vd.norm()
            valid_deltas[l] = vd
        else:
            valid_deltas[l] = deltas[l]

        if len(invalid_correct_acts[l]) > 0 and len(invalid_wrong_acts[l]) > 0:
            ivd = torch.stack(invalid_correct_acts[l]).mean(dim=0) - torch.stack(invalid_wrong_acts[l]).mean(dim=0)
            if NORMALIZE_VECTORS:
                ivd = ivd / ivd.norm()
            invalid_deltas[l] = ivd
        else:
            invalid_deltas[l] = deltas[l]

        print(f"  Layer {l}: ||valid_delta|| = {valid_deltas[l].norm():.4f}, ||invalid_delta|| = {invalid_deltas[l].norm():.4f}")

    cond_valid = {}
    cond_invalid = {}
    for l in target_layers:
        cond_valid[l] = torch.stack(valid_acts[l]).mean(dim=0)
        cond_invalid[l] = torch.stack(invalid_acts[l]).mean(dim=0)

    knn_store = {}
    for l in target_layers:
        vecs = torch.stack([item[0] for item in all_acts_with_validity[l]])
        labels = torch.tensor([item[1] for item in all_acts_with_validity[l]], dtype=torch.float32)
        knn_store[l] = {"vecs": vecs, "labels": labels}

    return {
        "deltas": deltas,
        "valid_deltas": valid_deltas,
        "invalid_deltas": invalid_deltas,
        "cond_valid": cond_valid,
        "cond_invalid": cond_invalid,
        "knn_store": knn_store,
    }


# ============================================================
# Hooks
# ============================================================

class StaticHook:
    def __init__(self, delta, alpha):
        self.delta = delta
        self.alpha = alpha

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            d = self.delta.to(x.device, x.dtype)
            return (x + self.alpha * d.view(1, 1, -1),) + output[1:]
        d = self.delta.to(output.device, output.dtype)
        return output + self.alpha * d.view(1, 1, -1)


class CASTHook:
    def __init__(self, delta, alpha, psi_valid, psi_invalid):
        self.delta = delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            effective_alpha = -self.alpha
        else:
            effective_alpha = self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class KCASTHook:
    def __init__(self, delta, alpha, knn_vecs, knn_labels, k):
        self.delta = delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        effective_alpha = -y_hat * self.alpha
        d = self.delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedHook:
    def __init__(self, valid_delta, invalid_delta, alpha, psi_valid, psi_invalid):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.psi_valid = psi_valid
        self.psi_invalid = psi_invalid

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        proj_valid = project_onto(h, self.psi_valid)
        proj_invalid = project_onto(h, self.psi_invalid)
        sim_valid = cosine_sim(h, proj_valid).item()
        sim_invalid = cosine_sim(h, proj_invalid).item()
        if sim_valid > sim_invalid:
            delta = self.valid_delta
            effective_alpha = -self.alpha
        else:
            delta = self.invalid_delta
            effective_alpha = self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


class ValidityConditionedKCASTHook:
    def __init__(self, valid_delta, invalid_delta, alpha, knn_vecs, knn_labels, k):
        self.valid_delta = valid_delta
        self.invalid_delta = invalid_delta
        self.alpha = alpha
        self.knn_vecs = knn_vecs
        self.knn_labels = knn_labels
        self.k = k

    def __call__(self, module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None
        h = x[0, -1, :].float().cpu()
        sims = F.cosine_similarity(h.unsqueeze(0), self.knn_vecs, dim=1)
        topk = torch.topk(sims, self.k)
        neighbor_labels = self.knn_labels[topk.indices]
        vote = neighbor_labels.sum().item()
        y_hat = 1.0 if vote > 0 else -1.0
        if y_hat > 0:
            delta = self.valid_delta
        else:
            delta = self.invalid_delta
        effective_alpha = -y_hat * self.alpha
        d = delta.to(x.device, x.dtype)
        x = x + effective_alpha * d.view(1, 1, -1)
        if rest is None:
            return x
        return (x,) + rest


def register_hooks(layers, steering_data, alpha, mode, layer_alphas=None):
    handles = []
    deltas = steering_data["deltas"]
    valid_deltas = steering_data["valid_deltas"]
    invalid_deltas = steering_data["invalid_deltas"]
    cond_valid = steering_data["cond_valid"]
    cond_invalid = steering_data["cond_invalid"]
    knn_store = steering_data["knn_store"]

    for layer_idx in deltas.keys():
        a = layer_alphas[layer_idx] if layer_alphas else alpha

        if mode == "static":
            hook = StaticHook(deltas[layer_idx], a)
        elif mode == "cast":
            hook = CASTHook(deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "kcast":
            hook = KCASTHook(deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode == "validity_cond":
            hook = ValidityConditionedHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, cond_valid[layer_idx], cond_invalid[layer_idx])
        elif mode == "validity_cond_kcast":
            hook = ValidityConditionedKCASTHook(valid_deltas[layer_idx], invalid_deltas[layer_idx], a, knn_store[layer_idx]["vecs"], knn_store[layer_idx]["labels"], KCAST_K)
        elif mode in ("layerwise", "layerwise_entropy", "entropy_gated", "structure_aware"):
            hook = StaticHook(deltas[layer_idx], a)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        handles.append(layers[layer_idx].register_forward_hook(hook))
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


# ============================================================
# Prediction functions
# ============================================================

@torch.no_grad()
def predict_dataset(model, tokenizer, train_items, eval_items, steering_data, alpha, mode, layer_alphas=None):
    layers = get_layers(model)

    if mode == "entropy_gated" or mode == "layerwise_entropy":
        return predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha, layer_alphas)

    handles = []
    if alpha != 0:
        handles = register_hooks(layers, steering_data, alpha, mode, layer_alphas)
    preds = []
    try:
        for i, ex in enumerate(eval_items):
            prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
            pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
            preds.append({"id": ex["id"], "validity": bool(pred)})
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(eval_items)}")
    finally:
        remove_hooks(handles)
    return preds


@torch.no_grad()
def predict_entropy_gated(model, tokenizer, train_items, eval_items, steering_data, alpha_base, layer_alphas=None):
    layers = get_layers(model)
    deltas = steering_data["deltas"]
    preds = []

    for i, ex in enumerate(eval_items):
        prompt = build_prompt(tokenizer, train_items, ex["syllogism"])
        norm_entropy, _ = get_prediction_entropy(model, tokenizer, prompt, MAX_LEN)

        if norm_entropy < ENTROPY_THRESHOLD:
            scale = (norm_entropy / ENTROPY_THRESHOLD) * 0.5
        else:
            scale = norm_entropy

        handles = []
        for layer_idx, delta in deltas.items():
            base_a = layer_alphas[layer_idx] if layer_alphas else alpha_base
            a = base_a * scale
            handles.append(layers[layer_idx].register_forward_hook(StaticHook(delta, a)))

        pred = predict_validity(model, tokenizer, prompt, MAX_LEN)
        remove_hooks(handles)

        preds.append({"id": ex["id"], "validity": bool(pred)})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_items)} (entropy_scale={scale:.3f})")

    return preds


# ============================================================
# Novelty 1: Layer-wise alpha optimization
# ============================================================

def optimize_layerwise_alphas(model, tokenizer, train_items, eval_items, steering_data, ref_path):
    target_layers = sorted(steering_data["deltas"].keys())
    n_layers = len(target_layers)
    alpha_vals = LAYERWISE_ALPHA_VALUES

    total_combos = len(alpha_vals) ** n_layers
    print(f"Layer-wise optimization: {n_layers} layers x {len(alpha_vals)} values = {total_combos} combos")

    if total_combos > LAYERWISE_MAX_COMBOS:
        print(f"Too many combos, using random search ({LAYERWISE_MAX_COMBOS} samples)")
        rng = random.Random(SEED)
        combos = []
        for _ in range(LAYERWISE_MAX_COMBOS):
            combo = tuple(rng.choice(alpha_vals) for _ in range(n_layers))
            combos.append(combo)
    else:
        combos = list(product(alpha_vals, repeat=n_layers))

    best_score = -1
    best_combo = None
    best_result = None

    for ci, combo in enumerate(combos):
        layer_alphas = {l: a for l, a in zip(target_layers, combo)}
        preds = predict_dataset(model, tokenizer, train_items, eval_items, steering_data, 0, "layerwise", layer_alphas)
        pred_path = os.path.join(OUT_DIR, "tmp_layerwise_preds.json")
        save_json(preds, pred_path)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "tmp_layerwise_eval.json"))
        score = result["combined_score"]

        if score > best_score:
            best_score = score
            best_combo = layer_alphas
            best_result = result
            print(f"  [{ci+1}/{len(combos)}] New best: {layer_alphas} -> Score={score:.4f} ACC={result['accuracy']:.2f} TCE={result['content_effect']:.4f}")

        if (ci + 1) % 100 == 0:
            print(f"  [{ci+1}/{len(combos)}] best so far: {best_score:.4f}")

    print(f"\nBest layer-wise alphas: {best_combo}")
    print(f"Best score: {best_score:.4f}")
    return best_combo, best_result


# ============================================================
# Evaluation helpers
# ============================================================

def print_subgroup_accuracy(items, preds):
    pred_map = {p["id"]: p["validity"] for p in preds}
    buckets = {}
    for item in items:
        v = "valid" if item["validity"] else "invalid"
        p = "plausible" if item.get("plausibility") else "implausible"
        key = f"{v}_{p}"
        buckets.setdefault(key, {"correct": 0, "total": 0})
        buckets[key]["total"] += 1
        if pred_map.get(item["id"]) == item["validity"]:
            buckets[key]["correct"] += 1
    for key in sorted(buckets.keys()):
        vals = buckets[key]
        acc = 100 * vals["correct"] / max(1, vals["total"])
        print(f"  {key:25s}: {acc:6.2f}%  ({vals['correct']}/{vals['total']})")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {MODEL_NAME}")
    print(f"Mode: {MODE}")
    print(f"Prompt: {PROMPT_MODE}" + (f" ({ICL_SHOTS} shots)" if PROMPT_MODE == "icl" else ""))
    print(f"Alphas: {ALPHAS}")
    if MODE == "kcast":
        print(f"K-CAST K={KCAST_K}")
    if MODE in ("entropy_gated", "layerwise_entropy"):
        print(f"Entropy threshold: {ENTROPY_THRESHOLD}")
    os.makedirs(OUT_DIR, exist_ok=True)

    token = HF_TOKEN if HF_TOKEN else None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if USE_BFLOAT16 and device == "cuda" else (torch.float16 if device == "cuda" else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None)
    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = False

    train_items = load_json(TRAIN_JSON)
    test_items = load_json(TEST_JSON)
    save_json(test_items, os.path.join(OUT_DIR, "test_reference.json"))
    ref_path = os.path.join(OUT_DIR, "test_reference.json")
    print(f"Train: {len(train_items)}, Test: {len(test_items)}")

    all_layers = get_layers(model)
    target_layers = get_target_layers(len(all_layers), LAYER_FRACTION)
    print(f"Total layers: {len(all_layers)}, Steering layers: {target_layers}")

    steering_data = compute_steering_data(model, tokenizer, train_items, target_layers)
    torch.save(steering_data, os.path.join(OUT_DIR, "steering_data.pt"))

    if MODE == "layerwise":
        best_layer_alphas, best_result = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": best_result}, os.path.join(OUT_DIR, "layerwise_best.json"))
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, 0, "layerwise", best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_best.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        return

    if MODE == "layerwise_entropy":
        best_layer_alphas, _ = optimize_layerwise_alphas(model, tokenizer, train_items, test_items, steering_data, ref_path)
        print(f"\nRunning entropy-gated with optimized per-layer alphas...")
        preds = predict_entropy_gated(model, tokenizer, train_items, test_items, steering_data, 0, best_layer_alphas)
        pred_path = os.path.join(OUT_DIR, "preds_layerwise_entropy.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, "eval_layerwise_entropy.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        save_json({"layer_alphas": {str(k): v for k, v in best_layer_alphas.items()}, "result": result}, os.path.join(OUT_DIR, "layerwise_entropy_best.json"))
        return

    best_score = -1
    best_alpha = 0
    summary = []

    for alpha in ALPHAS:
        print(f"\n===== {MODE.upper()} Alpha = {alpha} =====")
        preds = predict_dataset(model, tokenizer, train_items, test_items, steering_data, alpha, MODE)
        tag = str(alpha).replace("-", "neg").replace(".", "p")
        pred_path = os.path.join(OUT_DIR, f"preds_{MODE}_alpha_{tag}.json")
        save_json(preds, pred_path)
        print_subgroup_accuracy(test_items, preds)
        result = run_official_eval(EVAL_SCRIPT, ref_path, pred_path, os.path.join(OUT_DIR, f"eval_{MODE}_alpha_{tag}.json"))
        print(f"  ACC={result['accuracy']:.2f}  TCE={result['content_effect']:.4f}  Score={result['combined_score']:.4f}")
        summary.append({"alpha": alpha, "accuracy": result["accuracy"], "content_effect": result["content_effect"], "combined_score": result["combined_score"]})
        if result["combined_score"] > best_score:
            best_score = result["combined_score"]
            best_alpha = alpha

    print(f"\n===== SWEEP SUMMARY ({MODE.upper()}) =====")
    print(f"{'Alpha':>8s}  {'ACC':>8s}  {'TCE':>10s}  {'Score':>10s}")
    for s in summary:
        marker = " <-- best" if s["alpha"] == best_alpha else ""
        print(f"{s['alpha']:>8.1f}  {s['accuracy']:>8.2f}  {s['content_effect']:>10.4f}  {s['combined_score']:>10.4f}{marker}")
    print(f"\nBest alpha: {best_alpha} (combined_score: {best_score:.4f})")
    save_json(summary, os.path.join(OUT_DIR, f"sweep_summary_{MODE}.json"))


if __name__ == "__main__":
    main()

Device: cuda
Model: Qwen/Qwen2.5-7B-Instruct
Mode: kcast
Prompt: icl (4 shots)
Alphas: [-4, -3.5, -3, -2.5, -2]
K-CAST K=10


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Train: 960, Test: 191
Total layers: 28, Steering layers: [21, 22, 23, 24, 25, 26, 27]
Steering pool: 945 examples
  100/945 (correct=72, wrong=28)
  200/945 (correct=146, wrong=54)
  300/945 (correct=214, wrong=86)
  400/945 (correct=284, wrong=116)
  500/945 (correct=357, wrong=143)
  600/945 (correct=436, wrong=164)
  700/945 (correct=496, wrong=204)
  800/945 (correct=569, wrong=231)
  900/945 (correct=644, wrong=256)
  Total correct=677, wrong=268
  Layer 21: ||delta|| = 9.3561
  Layer 22: ||delta|| = 11.7454
  Layer 23: ||delta|| = 13.6107
  Layer 24: ||delta|| = 17.0605
  Layer 25: ||delta|| = 21.3400
  Layer 26: ||delta|| = 29.7729
  Layer 27: ||delta|| = 42.3385
  Layer 21: ||valid_delta|| = 45.5010, ||invalid_delta|| = 45.5742
  Layer 22: ||valid_delta|| = 67.0829, ||invalid_delta|| = 68.1453
  Layer 23: ||valid_delta|| = 100.0689, ||invalid_delta|| = 101.3446
  Layer 24: ||valid_delta|| = 112.8063, ||invalid_delta|| = 115.8288
  Layer 25: ||valid_delta|| = 125.1054, ||invalid